#### Taking extracted neuroimaging data and adding it to Scott 10K (local version so it runs on my laptop)
#### This only has joining logic of the various csvs (no_neuroimaging..., VBM data, SBM data)

### Add to scott_10k...

1. Desikan VBM ROIs
2. Desikan coeffs
3. Desikan regression stats
4. Desikan flipped coeffs
5. Desikan  flipped regression stats


1. Destrieux VBM ROIs
2. Destrieux coeffs
3. Destrieux regression stats
4. Destrieux flipped coeffs
5. Destrieux flipped regression stats


1. Aseg stats (volume)
2. Desikan volume (lh, rh)
3. Destrieux volume (lh, rh)
4. Desikan thickness (lh, rh)
5. Destrieux volume (lh, rh)

In [1]:
import pandas as pd, pandasql as ps, os, re

df = pd.read_csv('C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_housekeeping/no_neuroimaging_data_scott10k_alliedhealth.csv', low_memory = False)

#Add VBM files (files 1-10)
path = 'C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_housekeeping/'
selected_files = ['gm_only_scraped_desikan_stats', 'desikan_coeffs', 'desikan_stats', 'flipped_desikan_coeffs', 'flipped_desikan_stats',
                 'gm_only_scraped_destrieux_stats', 'destrieux_coeffs', 'destrieux_stats', 'flipped_destrieux_coeffs', 'flipped_destrieux_stats']

for file in selected_files:
        vbm_path = os.path.join(path, f'{file}.csv')
        vbm_df = pd.read_csv(vbm_path, low_memory = False)
        file_name_parts  = file.split('_')
        if file in ['desikan_stats', 'destrieux_stats', 'flipped_desikan_stats', 'flipped_destrieux_stats']:
            vbm_df = vbm_df.loc[:, ~vbm_df.columns.str.contains('^Unnamed')]           
            col_prefix = '_'.join(file_name_parts[:-1]) + '_'
            excluded = ['DATA_KEY']
            vbm_df.rename(columns = lambda x: col_prefix + x if x not in excluded else x, inplace = True)
        elif file in ['gm_only_scraped_desikan_stats']:
            vbm_df.drop(columns=['Unnamed: 0'], errors='ignore', inplace=True)            
            col_prefix = 'desikan_vbm_'
            excluded = ['DATA_KEY']
            vbm_df.rename(columns = lambda x: col_prefix + x if x not in excluded else x, inplace = True)
        elif file in ['gm_only_scraped_destrieux_stats']:
            vbm_df = vbm_df.loc[:, ~vbm_df.columns.str.contains('^Unnamed')]           
            col_prefix = 'destrieux_vbm_'
            excluded = ['DATA_KEY']
            vbm_df.rename(columns = lambda x: col_prefix + x if x not in excluded else x, inplace = True)
        df = pd.merge(df, vbm_df, how = 'left', on = 'DATA_KEY')
        print(df.shape, end = ' ')
df.to_csv('C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_housekeeping/w_vbm_scott10k_alliedhealth.csv', index = False)    
print(df.columns.tolist())

(15733, 163) (15733, 182) (15733, 185) (15733, 204) (15733, 207) (15733, 373) (15733, 392) (15733, 395) (15733, 414) (15733, 417) ['PTID', 'RID', 'SITE_ID', 'DATA_KEY', 'T1_YEAR', 'T1_MON', 'T1_DAY', 'VISDATE', 'EXAMDATE_4WKS_LATER', 'EXAMDATE_4WKS_B4', 'T1_PATH', 'MWC1T1_PATH', 'PTGENDER', 'PTDOB', 'PTEDUCAT', 'PTAGE', 'GENOTYPE', 'EXAMDATE_MRIFLDSTRNGTH', 'FIELD_STRENGTH', 'DATEDIFFS_MRIFLDSTRNGTH', 'EXAMDATE_DXSUM', 'PHASE', 'DIAGNOSIS', 'DATEDIFFS_DXSUM', 'EXAMDATE_MMSE', 'MMSCORE', 'DATEDIFFS_MMSE', 'EXAMDATE_MOCA', 'MOCA', 'DATEDIFFS_MOCA', 'EXAMDATE_NEUROBAT', 'LIMMTOTAL', 'CLOCKSCOR', 'LDELTOTAL', 'LDELCUE', 'ANART', 'DATEDIFFS_NEUROBAT', 'EXAMDATE_CDR', 'CDGLOBAL', 'DATEDIFFS_CDR', 'EXAMDATE_FAQ', 'FAQTOTAL', 'DATEDIFFS_FAQ', 'EXAMDATE_NPIQ', 'NPISCORE', 'DATEDIFFS_NPIQ', 'EXAMDATE_GDSCALE', 'GDTOTAL', 'DATEDIFFS_GDSCALE', 'EXAMDATE_MODHACH', 'HMSCORE', 'DATEDIFFS_MODHACH', 'EXAMDATE_ADAS13', 'TOTAL13', 'DATEDIFFS_ADAS13', 'EXAMDATE_P217', 'P217_DILUTION_CORRECTED_CONC', 'P217

In [10]:
import pandas as pd, pandasql as ps, os

df = pd.read_csv('C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_housekeeping/w_vbm_scott10k_alliedhealth.csv', low_memory = False)    
print(df.shape)
path = 'C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_reconall_scraped/'
selected_files = ['aparc_thickness_lh.csv','aparc_thickness_rh.csv', # Desikan thickness
                    'lh.a2009s.thickness.csv', 'rh.a2009s.thickness.csv', # Destrieux thickness
                     'aparc_volume_lh.csv', 'aparc_volume_rh.csv', # Desikan volume
                     'lh.a2009s.volume.csv', 'rh.a2009s.volume.csv', # Destrieux volume
                    'aseg_stats.csv'] # ASEG volume
        

for file in selected_files:
    try:
        sbm_path = os.path.join(path,f'z_norm_{file}')
        if os.path.exists(sbm_path):
            sbm_df = pd.read_csv(sbm_path, low_memory = False)
        df = pd.merge(df, sbm_df, how = 'left', on = ['DATA_KEY', 'DIAGNOSIS'], suffixes = (None, None))
        print(df.shape)
    except Exception:
        raise Exception
df.to_csv('C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_housekeeping/sbm_w_vbm_scott10k_alliedhealth.csv', index = False)  
print(df.columns.tolist())

(15733, 417)
(15733, 454)
(15733, 491)
(15733, 568)
(15733, 645)
(15733, 680)
(15733, 715)
(15733, 790)
(15733, 865)
(15733, 928)
['PTID', 'RID', 'SITE_ID', 'DATA_KEY', 'T1_YEAR', 'T1_MON', 'T1_DAY', 'VISDATE', 'EXAMDATE_4WKS_LATER', 'EXAMDATE_4WKS_B4', 'T1_PATH', 'MWC1T1_PATH', 'PTGENDER', 'PTDOB', 'PTEDUCAT', 'PTAGE', 'GENOTYPE', 'EXAMDATE_MRIFLDSTRNGTH', 'FIELD_STRENGTH', 'DATEDIFFS_MRIFLDSTRNGTH', 'EXAMDATE_DXSUM', 'PHASE', 'DIAGNOSIS', 'DATEDIFFS_DXSUM', 'EXAMDATE_MMSE', 'MMSCORE', 'DATEDIFFS_MMSE', 'EXAMDATE_MOCA', 'MOCA', 'DATEDIFFS_MOCA', 'EXAMDATE_NEUROBAT', 'LIMMTOTAL', 'CLOCKSCOR', 'LDELTOTAL', 'LDELCUE', 'ANART', 'DATEDIFFS_NEUROBAT', 'EXAMDATE_CDR', 'CDGLOBAL', 'DATEDIFFS_CDR', 'EXAMDATE_FAQ', 'FAQTOTAL', 'DATEDIFFS_FAQ', 'EXAMDATE_NPIQ', 'NPISCORE', 'DATEDIFFS_NPIQ', 'EXAMDATE_GDSCALE', 'GDTOTAL', 'DATEDIFFS_GDSCALE', 'EXAMDATE_MODHACH', 'HMSCORE', 'DATEDIFFS_MODHACH', 'EXAMDATE_ADAS13', 'TOTAL13', 'DATEDIFFS_ADAS13', 'EXAMDATE_P217', 'P217_DILUTION_CORRECTED_CONC', 'P217

In [16]:
import pandas as pd, os, numpy as np

df = pd.read_csv('C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_housekeeping/sbm_w_vbm_scott10k_alliedhealth.csv', low_memory = False)

df['PTDOB'] = pd.to_datetime(
    df['PTDOB'],
    format='%Y-%m-%d',
    errors='coerce'
)

df['VISDATE'] = pd.to_datetime(
    df['VISDATE'],
    format='%Y-%m-%d',
    errors='coerce'
)

In [17]:


df['PTDOB'] = (
    df.groupby('PTID')['PTDOB']
      .transform(lambda x: x.ffill().bfill())
)

df['PTDOB'] = (
    df.groupby('PTID')['PTDOB']
      .transform(lambda x: x.ffill().bfill())
)

df['PTEDUCAT'] = (
    df.groupby('PTID')['PTEDUCAT']
      .transform(lambda x: x.ffill().bfill())
)

df['PTAGE'] = np.floor((df['VISDATE'] - df['PTDOB']).dt.days / 365.25)

df.to_csv('C:/Users/User/Downloads/deconstructalz.github.io/scott_10k_housekeeping/sbm_w_vbm_scott10k_alliedhealth.csv', index = False)  
